# ============================================================
# Phase 05A - BTUMQA-225K PRUGTM-Hybrid Three-Seed Q-CUR Reliability
# PRUGTM-Hybrid main-reference reliability calibration over seed_42, seed_1337, and seed_2025 Phase 4 outputs
# ============================================================


# Install Required Libraries


In [1]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas",
        "scikit-learn",
        "tqdm",
        "matplotlib",
        "seaborn",
        "joblib",
    ],
    check=True,
)
print("Required libraries are installed.")


Required libraries are installed.


# Mount Google Drive and Configure Phase 05A PRUGTM-Hybrid Q-CUR Paths


In [2]:
from google.colab import drive
from pathlib import Path
import os
import time


def ensure_drive_connection(project_dir: Path, mount_point: str = "/content/drive"):
    def probe_path(target: Path):
        probe_target = target if target.exists() else target.parent
        return os.listdir(str(probe_target))

    try:
        probe_path(project_dir)
    except OSError as exc:
        if getattr(exc, "errno", None) != 107:
            raise
        print("Detected a stale Google Drive mount. Remounting now...")
        try:
            drive.flush_and_unmount()
            time.sleep(2)
        except Exception:
            pass
        drive.mount(mount_point, force_remount=True)
        time.sleep(2)
        probe_path(project_dir)

    if not project_dir.exists():
        raise FileNotFoundError(
            f"Project Drive directory not found after mount check: {project_dir}"
        )


drive.mount("/content/drive")

# Updated project path to point to the new drive structure
PROJECT_DRIVE_DIR = Path("/content/drive/MyDrive/AUGR-VQA")

ensure_drive_connection(PROJECT_DRIVE_DIR)

DATASET_RELEASE_NAME = "BTUMQA-225K"
PHASE5_METHOD_NAME = "BrainTumorVQA-BTUMQA-225K-PRUGTM-Hybrid-QCUR-MainReference-FourSeed"
PHASE5_MODEL_VARIANT = "btumqa_225k_clean_metadata_qgca_qcur_main_reference_four_seed"

PHASE4_REFERENCE_PARENT_DIR = PROJECT_DRIVE_DIR / "phase_4" / "p4a_qgca_training" / "clean_metadata_training" / "adaptive_prugtm_qgca_btumqa_four_seeds"
PHASE5_BASE_DIR = PROJECT_DRIVE_DIR / "phase_5" / "p5a_proposed_model_qcur_reliability"
PHASE5_DIR = PHASE5_BASE_DIR / PHASE5_MODEL_VARIANT
MODELS_DIR = PHASE5_DIR / "models"
REPORTS_DIR = PHASE5_DIR / "reports"
PREDICTIONS_DIR = PHASE5_DIR / "predictions"
FIGURES_DIR = PHASE5_DIR / "figures"
DONE_DIR = PHASE5_DIR / "done"

for path in [PHASE5_DIR, MODELS_DIR, REPORTS_DIR, PREDICTIONS_DIR, FIGURES_DIR, DONE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

QCUR_CONFIG_PATH = PHASE5_DIR / "phase05a_prugtm_hybrid_qcur_four_seed_config.json"
QCUR_REPORT_PATH = PHASE5_DIR / "phase05a_prugtm_hybrid_qcur_four_seed_report.json"
QCUR_AGGREGATE_SUMMARY_PATH = REPORTS_DIR / "qcur_prugtm_hybrid_four_seed_method_summary.json"
QCUR_AGGREGATE_METHOD_TABLE_PATH = REPORTS_DIR / "qcur_prugtm_hybrid_four_seed_method_table.csv"
QCUR_PER_SEED_METHOD_TABLE_PATH = REPORTS_DIR / "qcur_prugtm_hybrid_four_seed_per_seed_method_table.csv"
QCUR_AGGREGATE_DONE_PATH = DONE_DIR / "phase05a_prugtm_hybrid_qcur_four_seed_complete.json"

print("Project directory:", PROJECT_DRIVE_DIR)
print("Phase 4 PRUGTM-Hybrid main reference parent dir:", PHASE4_REFERENCE_PARENT_DIR)
print("Phase 5A Q-CUR dir:", PHASE5_DIR)
print("Phase 4 parent exists:", PHASE4_REFERENCE_PARENT_DIR.exists())

Mounted at /content/drive
Project directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging
Phase 4 PRUGTM-Hybrid main reference parent dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/single_stage_btumqa_225k_prugtm_hybrid_45e
Phase 5A Q-CUR dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed
Phase 4 parent exists: True


# Phase 05A PRUGTM-Hybrid Q-CUR Configuration


In [3]:
RUN_SEEDS = [42, 1337, 2025, 3407]
RANDOM_STATE = 42
CALIBRATION_BINS = 15
SELECTIVE_COVERAGES = [0.50, 0.70, 0.80, 0.90, 0.95, 1.00]
ENABLE_OPTIONAL_HYBRID_RAW_COMPARISON = False
PLOT_FIGURES = True
MIN_CONFIDENCE = 1e-6

BASELINE_NUMERIC_FEATURES = ["confidence", "confidence_logit"]
QCUR_NUMERIC_FEATURES = ["confidence", "confidence_logit"]
QCUR_CATEGORICAL_FEATURES = [
    "predicted_answer",
    "question_family",
    "question_style",
    "difficulty_level",
    "ambiguity_flag",
    "signal_gap_bucket",
    "region_target_pair",
]

print("Run seeds:", RUN_SEEDS)
print("Random state:", RANDOM_STATE)
print("Calibration bins:", CALIBRATION_BINS)
print("Selective coverages:", SELECTIVE_COVERAGES)
print("Optional hybrid raw comparison:", ENABLE_OPTIONAL_HYBRID_RAW_COMPARISON)
print("Q-CUR categorical features:", QCUR_CATEGORICAL_FEATURES)

Run seeds: [42, 1337, 2025]
Random state: 42
Calibration bins: 15
Selective coverages: [0.5, 0.7, 0.8, 0.9, 0.95, 1.0]
Optional hybrid raw comparison: False
Q-CUR categorical features: ['predicted_answer', 'question_family', 'question_style', 'difficulty_level', 'ambiguity_flag', 'signal_gap_bucket', 'region_target_pair']


# Imports and Helper Utilities


In [4]:
import csv
import json
import math
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")


def now_string():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def read_json(path: Path, default=None):
    if not path.exists():
        return {} if default is None else default
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def write_csv(path: Path, rows, fieldnames):
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def clamp_probs(values, eps=MIN_CONFIDENCE):
    arr = np.asarray(values, dtype=np.float64)
    return np.clip(arr, eps, 1.0 - eps)


def safe_logit(values):
    p = clamp_probs(values)
    return np.log(p / (1.0 - p))


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def compute_answer_metrics(df: pd.DataFrame):
    y_true = df["gold_answer"].astype(str).to_numpy()
    y_pred = df["predicted_answer"].astype(str).to_numpy()
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }


def compute_ece(y_true, y_prob, n_bins=CALIBRATION_BINS):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_prob = clamp_probs(y_prob)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []
    for idx in range(n_bins):
        left = bin_edges[idx]
        right = bin_edges[idx + 1]
        if idx == n_bins - 1:
            mask = (y_prob >= left) & (y_prob <= right)
        else:
            mask = (y_prob >= left) & (y_prob < right)
        count = int(mask.sum())
        if count == 0:
            rows.append({
                "bin_index": idx,
                "bin_left": float(left),
                "bin_right": float(right),
                "count": 0,
                "mean_confidence": None,
                "accuracy": None,
                "gap": None,
            })
            continue
        mean_conf = float(y_prob[mask].mean())
        mean_acc = float(y_true[mask].mean())
        gap = abs(mean_conf - mean_acc)
        ece += (count / len(y_true)) * gap
        rows.append({
            "bin_index": idx,
            "bin_left": float(left),
            "bin_right": float(right),
            "count": count,
            "mean_confidence": mean_conf,
            "accuracy": mean_acc,
            "gap": float(gap),
        })
    return float(ece), rows


def compute_selective_accuracy(y_true, y_prob, coverages=SELECTIVE_COVERAGES):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    order = np.argsort(-y_prob)
    y_sorted = y_true[order]
    result = {}
    for coverage in coverages:
        k = max(1, int(math.ceil(len(y_sorted) * float(coverage))))
        result[f"{coverage:.2f}"] = float(y_sorted[:k].mean())
    return result


def compute_aurc(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    order = np.argsort(-y_prob)
    correct_sorted = y_true[order]
    cumulative_correct = np.cumsum(correct_sorted)
    counts = np.arange(1, len(correct_sorted) + 1, dtype=np.float64)
    coverage = counts / len(correct_sorted)
    risk = 1.0 - (cumulative_correct / counts)
    # aurc = float(np.trapz(risk, coverage))
    try:
        aurc = float(np.trapezoid(risk, coverage))
    except AttributeError:
        aurc = float(np.trapz(risk, coverage))
    return aurc, coverage, risk


def compute_calibration_metrics(df: pd.DataFrame, prob_col: str):
    y_true = df["is_correct"].astype(int).to_numpy()
    y_prob = clamp_probs(df[prob_col].astype(float).to_numpy())
    ece, reliability_rows = compute_ece(y_true, y_prob)
    brier = float(np.mean((y_prob - y_true) ** 2))
    nll = float(log_loss(y_true, y_prob, labels=[0, 1]))
    aurc, coverage, risk = compute_aurc(y_true, y_prob)
    selective_accuracy = compute_selective_accuracy(y_true, y_prob)
    return {
        "ece": ece,
        "brier": brier,
        "nll": nll,
        "aurc": aurc,
        "selective_accuracy": selective_accuracy,
        "reliability_bins": reliability_rows,
        "risk_coverage_curve": {
            "coverage": [float(x) for x in coverage.tolist()],
            "risk": [float(x) for x in risk.tolist()],
        },
    }


def summarize_method(df: pd.DataFrame, prob_col: str, method_name: str):
    answer_metrics = compute_answer_metrics(df)
    calib = compute_calibration_metrics(df, prob_col)
    return {
        "method_name": method_name,
        "num_rows": int(len(df)),
        "answer_metrics": answer_metrics,
        "calibration_metrics": {
            "ece": float(calib["ece"]),
            "brier": float(calib["brier"]),
            "nll": float(calib["nll"]),
            "aurc": float(calib["aurc"]),
            "selective_accuracy": calib["selective_accuracy"],
        },
        "reliability_bins": calib["reliability_bins"],
        "risk_coverage_curve": calib["risk_coverage_curve"],
    }


def compute_groupwise_method_metrics(df: pd.DataFrame, prob_col: str, group_col: str):
    grouped = {}
    for group_value, group_df in df.groupby(group_col, dropna=False):
        label = str(group_value) if str(group_value) != "" else "<empty>"
        grouped[label] = summarize_method(group_df.reset_index(drop=True), prob_col, label)
    return grouped


def build_method_table(method_summaries: dict):
    rows = []
    for name, summary in method_summaries.items():
        row = {
            "method": name,
            "num_rows": summary["num_rows"],
            "accuracy": summary["answer_metrics"]["accuracy"],
            "macro_f1": summary["answer_metrics"]["macro_f1"],
            "weighted_f1": summary["answer_metrics"]["weighted_f1"],
            "ece": summary["calibration_metrics"]["ece"],
            "brier": summary["calibration_metrics"]["brier"],
            "nll": summary["calibration_metrics"]["nll"],
            "aurc": summary["calibration_metrics"]["aurc"],
        }
        for coverage_key, value in summary["calibration_metrics"]["selective_accuracy"].items():
            row[f"selective_acc_at_{coverage_key}"] = value
        rows.append(row)
    return pd.DataFrame(rows)



def seed_tag(run_seed: int):
    return f"seed_{int(run_seed)}"


def build_phase4_seed_paths(run_seed: int):
    tag = seed_tag(run_seed)
    phase4_dir = PHASE4_REFERENCE_PARENT_DIR / tag
    return {
        "run_seed": int(run_seed),
        "seed_tag": tag,
        "phase4_dir": phase4_dir,
        "val_predictions_input_path": phase4_dir / "predictions" / "val_predictions.csv",
        "test_predictions_input_path": phase4_dir / "predictions" / "test_predictions.csv",
        "phase4_val_metrics_path": phase4_dir / "reports" / "val_metrics.json",
        "phase4_test_metrics_path": phase4_dir / "reports" / "test_metrics.json",
        "phase4_final_eval_path": phase4_dir / "reports" / "final_eval_summary.json",
    }


def build_phase5_seed_paths(run_seed: int):
    tag = seed_tag(run_seed)
    seed_dir = PHASE5_DIR / tag
    models_dir = seed_dir / "models"
    reports_dir = seed_dir / "reports"
    predictions_dir = seed_dir / "predictions"
    figures_dir = seed_dir / "figures"
    done_dir = seed_dir / "done"
    for path in [seed_dir, models_dir, reports_dir, predictions_dir, figures_dir, done_dir]:
        path.mkdir(parents=True, exist_ok=True)
    return {
        "run_seed": int(run_seed),
        "seed_tag": tag,
        "phase5_seed_dir": seed_dir,
        "models_dir": models_dir,
        "reports_dir": reports_dir,
        "predictions_dir": predictions_dir,
        "figures_dir": figures_dir,
        "done_dir": done_dir,
        "qcur_config_path": seed_dir / "phase05a_qcur_config.json",
        "qcur_report_path": seed_dir / "phase05a_prugtm_hybrid_qcur_report.json",
        "qcur_summary_path": reports_dir / "qcur_method_summary.json",
        "qcur_slice_metrics_path": reports_dir / "qcur_slice_metrics.json",
        "qcur_method_table_path": reports_dir / "qcur_method_table.csv",
        "qcur_val_predictions_path": predictions_dir / "val_qcur_predictions.csv",
        "qcur_test_predictions_path": predictions_dir / "test_qcur_predictions.csv",
        "qcur_models_path": models_dir / "qcur_models.joblib",
        "qcur_reliability_plot_path": figures_dir / "qcur_reliability_diagram.png",
        "qcur_risk_coverage_plot_path": figures_dir / "qcur_risk_coverage.png",
        "qcur_done_path": done_dir / "phase05a_qcur_complete.json",
    }


def aggregate_method_tables(seed_tables: list):
    combined = pd.concat(seed_tables, ignore_index=True)
    metric_cols = [col for col in combined.columns if col not in {"run_seed", "seed_tag", "method"}]
    aggregate_rows = []
    for method, method_df in combined.groupby("method", sort=False):
        row = {
            "method": method,
            "num_runs": int(method_df["run_seed"].nunique()),
        }
        for col in metric_cols:
            values = method_df[col].astype(float)
            row[f"{col}_mean"] = float(values.mean())
            row[f"{col}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
        aggregate_rows.append(row)
    return combined, pd.DataFrame(aggregate_rows)

print("Multi-seed helper utilities ready.")


Multi-seed helper utilities ready.


# Load Phase 4 PRUGTM-Hybrid Three-Seed Main Reference Predictions


In [5]:
REQUIRED_COLUMNS = [
    "split",
    "qa_id",
    "unique_id",
    "question_family",
    "question_style",
    "difficulty_level",
    "ambiguity_flag",
    "signal_gap_bucket",
    "region_target_primary",
    "region_target_secondary",
    "region_target_pair",
    "gold_answer",
    "predicted_answer",
    "confidence",
]


def load_predictions_csv(path: Path, split_name: str):
    if not path.exists():
        raise FileNotFoundError(f"{split_name} predictions not found: {path}")
    df = pd.read_csv(path, dtype=str).fillna("")
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{split_name} predictions missing required columns: {missing}")
    df["confidence"] = df["confidence"].astype(float).clip(MIN_CONFIDENCE, 1.0 - MIN_CONFIDENCE)
    df["confidence_logit"] = safe_logit(df["confidence"].to_numpy())
    df["is_correct"] = (df["gold_answer"] == df["predicted_answer"]).astype(int)
    return df


SEED_RUNS = {}

for run_seed in RUN_SEEDS:
    tag = seed_tag(run_seed)
    phase4_paths = build_phase4_seed_paths(run_seed)
    phase5_paths = build_phase5_seed_paths(run_seed)

    val_df = load_predictions_csv(phase4_paths["val_predictions_input_path"], f"{tag} val")
    test_df = load_predictions_csv(phase4_paths["test_predictions_input_path"], f"{tag} test")

    SEED_RUNS[tag] = {
        "run_seed": int(run_seed),
        "seed_tag": tag,
        "phase4_paths": phase4_paths,
        "phase5_paths": phase5_paths,
        "val_df": val_df,
        "test_df": test_df,
        "phase4_val_metrics": read_json(phase4_paths["phase4_val_metrics_path"]),
        "phase4_test_metrics": read_json(phase4_paths["phase4_test_metrics_path"]),
        "phase4_final_eval": read_json(phase4_paths["phase4_final_eval_path"]),
    }

    print("\n" + "=" * 100)
    print(f"Loaded {tag}")
    print("Phase 4 dir:", phase4_paths["phase4_dir"])
    print("Phase 5A seed dir:", phase5_paths["phase5_seed_dir"])
    print("Validation rows:", len(val_df))
    print("Test rows:", len(test_df))
    print("Validation correctness rate:", round(float(val_df["is_correct"].mean()), 6))
    print("Test correctness rate:", round(float(test_df["is_correct"].mean()), 6))



Loaded seed_42
Phase 4 dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/single_stage_btumqa_225k_prugtm_hybrid_45e/seed_42
Phase 5A seed dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_42
Validation rows: 33750
Test rows: 33750
Validation correctness rate: 0.943822
Test correctness rate: 0.946074

Loaded seed_1337
Phase 4 dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/single_stage_btumqa_225k_prugtm_hybrid_45e/seed_1337
Phase 5A seed dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_1337
Validation rows: 33750
Test ro

# Train Per-Seed PRUGTM-Hybrid Confidence Baselines and Learned Q-CUR Models


In [6]:
for tag, run in SEED_RUNS.items():
    run_seed = run["run_seed"]
    phase5_paths = run["phase5_paths"]
    val_df = run["val_df"]
    test_df = run["test_df"]

    baseline_model = Pipeline(
        steps=[
            ("preprocess", ColumnTransformer(
                transformers=[("numeric", StandardScaler(), BASELINE_NUMERIC_FEATURES)],
                remainder="drop",
            )),
            ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE + int(run_seed))),
        ]
    )

    qcur_model = Pipeline(
        steps=[
            ("preprocess", ColumnTransformer(
                transformers=[
                    ("numeric", StandardScaler(), QCUR_NUMERIC_FEATURES),
                    ("categorical", make_one_hot_encoder(), QCUR_CATEGORICAL_FEATURES),
                ],
                remainder="drop",
            )),
            ("classifier", LogisticRegression(max_iter=2500, random_state=RANDOM_STATE + int(run_seed))),
        ]
    )

    y_val = val_df["is_correct"].astype(int).to_numpy()

    baseline_model.fit(val_df[BASELINE_NUMERIC_FEATURES], y_val)
    qcur_model.fit(val_df[QCUR_NUMERIC_FEATURES + QCUR_CATEGORICAL_FEATURES], y_val)

    val_df = val_df.copy()
    test_df = test_df.copy()

    val_df["raw_confidence"] = val_df["confidence"].astype(float)
    test_df["raw_confidence"] = test_df["confidence"].astype(float)
    val_df["posthoc_confidence"] = baseline_model.predict_proba(val_df[BASELINE_NUMERIC_FEATURES])[:, 1]
    test_df["posthoc_confidence"] = baseline_model.predict_proba(test_df[BASELINE_NUMERIC_FEATURES])[:, 1]
    val_df["qcur_confidence"] = qcur_model.predict_proba(
        val_df[QCUR_NUMERIC_FEATURES + QCUR_CATEGORICAL_FEATURES]
    )[:, 1]
    test_df["qcur_confidence"] = qcur_model.predict_proba(
        test_df[QCUR_NUMERIC_FEATURES + QCUR_CATEGORICAL_FEATURES]
    )[:, 1]

    joblib.dump(
        {
            "baseline_model": baseline_model,
            "qcur_model": qcur_model,
            "baseline_numeric_features": BASELINE_NUMERIC_FEATURES,
            "qcur_numeric_features": QCUR_NUMERIC_FEATURES,
            "qcur_categorical_features": QCUR_CATEGORICAL_FEATURES,
            "run_seed": int(run_seed),
            "seed_tag": tag,
        },
        phase5_paths["qcur_models_path"],
    )

    run["val_df"] = val_df
    run["test_df"] = test_df
    run["baseline_model"] = baseline_model
    run["qcur_model"] = qcur_model

    print(f"{tag}: baseline and Q-CUR models trained and saved: {phase5_paths['qcur_models_path']}")


seed_42: baseline and Q-CUR models trained and saved: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_42/models/qcur_models.joblib
seed_1337: baseline and Q-CUR models trained and saved: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_1337/models/qcur_models.joblib
seed_2025: baseline and Q-CUR models trained and saved: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_2025/models/qcur_models.joblib


# Evaluate Per-Seed Raw Confidence, Post-Hoc Baseline, and Q-CUR


In [7]:
SEED_METHOD_TABLES = []

for tag, run in SEED_RUNS.items():
    phase5_paths = run["phase5_paths"]
    test_df = run["test_df"]

    method_summaries = {
        "main_raw_confidence": summarize_method(test_df, "raw_confidence", "main_raw_confidence"),
        "main_posthoc_confidence_only": summarize_method(test_df, "posthoc_confidence", "main_posthoc_confidence_only"),
        "main_qcur": summarize_method(test_df, "qcur_confidence", "main_qcur"),
    }

    slice_metrics = {}
    for group_name, group_col in [
        ("by_question_style", "question_style"),
        ("by_question_family", "question_family"),
        ("by_ambiguity_flag", "ambiguity_flag"),
        ("by_signal_gap_bucket", "signal_gap_bucket"),
        ("by_region_target_pair", "region_target_pair"),
    ]:
        slice_metrics[group_name] = {
            "main_raw_confidence": compute_groupwise_method_metrics(test_df, "raw_confidence", group_col),
            "main_posthoc_confidence_only": compute_groupwise_method_metrics(test_df, "posthoc_confidence", group_col),
            "main_qcur": compute_groupwise_method_metrics(test_df, "qcur_confidence", group_col),
        }

    method_table_df = build_method_table(method_summaries)
    method_table_df.insert(0, "seed_tag", tag)
    method_table_df.insert(0, "run_seed", run["run_seed"])
    method_table_df.to_csv(phase5_paths["qcur_method_table_path"], index=False)
    write_json(phase5_paths["qcur_summary_path"], method_summaries)
    write_json(phase5_paths["qcur_slice_metrics_path"], slice_metrics)

    run["method_summaries"] = method_summaries
    run["slice_metrics"] = slice_metrics
    run["method_table_df"] = method_table_df
    SEED_METHOD_TABLES.append(method_table_df)

    print("\n" + "=" * 100)
    print(f"{tag} method table:")
    print(method_table_df.round(6).to_string(index=False))
    print("Method summary saved:", phase5_paths["qcur_summary_path"])
    print("Slice metrics saved:", phase5_paths["qcur_slice_metrics_path"])
    print("Method table saved:", phase5_paths["qcur_method_table_path"])

SEED_METHOD_TABLE_DF, AGGREGATE_METHOD_TABLE_DF = aggregate_method_tables(SEED_METHOD_TABLES)
SEED_METHOD_TABLE_DF.to_csv(QCUR_PER_SEED_METHOD_TABLE_PATH, index=False)
AGGREGATE_METHOD_TABLE_DF.to_csv(QCUR_AGGREGATE_METHOD_TABLE_PATH, index=False)

AGGREGATE_METHOD_SUMMARY = {
    "created_at": now_string(),
    "run_seeds": [int(seed) for seed in RUN_SEEDS],
    "num_runs": int(len(RUN_SEEDS)),
    "per_seed_method_table_path": str(QCUR_PER_SEED_METHOD_TABLE_PATH),
    "aggregate_method_table_path": str(QCUR_AGGREGATE_METHOD_TABLE_PATH),
    "aggregate_rows": AGGREGATE_METHOD_TABLE_DF.to_dict(orient="records"),
}
write_json(QCUR_AGGREGATE_SUMMARY_PATH, AGGREGATE_METHOD_SUMMARY)

print("\nAggregate method table:")
print(AGGREGATE_METHOD_TABLE_DF.round(6).to_string(index=False))
print("Aggregate method summary saved:", QCUR_AGGREGATE_SUMMARY_PATH)
print("Aggregate method table saved:", QCUR_AGGREGATE_METHOD_TABLE_PATH)



seed_42 method table:
 run_seed seed_tag                       method  num_rows  accuracy  macro_f1  weighted_f1      ece    brier      nll     aurc  selective_acc_at_0.50  selective_acc_at_0.70  selective_acc_at_0.80  selective_acc_at_0.90  selective_acc_at_0.95  selective_acc_at_1.00
       42  seed_42          main_raw_confidence     33750  0.946074  0.972353     0.946125 0.150283 0.060081 0.247349 0.004806               0.998459               0.998561               0.998481               0.987753               0.968468               0.946074
       42  seed_42 main_posthoc_confidence_only     33750  0.946074  0.972353     0.946125 0.012688 0.033839 0.109638 0.004747               0.998459               0.998561               0.998481               0.987753               0.968468               0.946074
       42  seed_42                    main_qcur     33750  0.946074  0.972353     0.946125 0.002437 0.032600 0.095594 0.003451               1.000000               1.000000          

# Save Per-Seed Per-Row Reliability Predictions


In [8]:
prediction_columns = [
    "split",
    "qa_id",
    "unique_id",
    "question_family",
    "question_style",
    "difficulty_level",
    "ambiguity_flag",
    "signal_gap_bucket",
    "region_target_primary",
    "region_target_secondary",
    "region_target_pair",
    "gold_answer",
    "predicted_answer",
    "is_correct",
    "raw_confidence",
    "posthoc_confidence",
    "qcur_confidence",
]

for tag, run in SEED_RUNS.items():
    phase5_paths = run["phase5_paths"]
    run["val_df"][prediction_columns].to_csv(phase5_paths["qcur_val_predictions_path"], index=False)
    run["test_df"][prediction_columns].to_csv(phase5_paths["qcur_test_predictions_path"], index=False)
    print(f"{tag}: saved validation reliability predictions: {phase5_paths['qcur_val_predictions_path']}")
    print(f"{tag}: saved test reliability predictions: {phase5_paths['qcur_test_predictions_path']}")


seed_42: saved validation reliability predictions: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_42/predictions/val_qcur_predictions.csv
seed_42: saved test reliability predictions: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_42/predictions/test_qcur_predictions.csv
seed_1337: saved validation reliability predictions: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_1337/predictions/val_qcur_predictions.csv
seed_1337: saved test reliability predictions: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tu

# Save Per-Seed Reliability Plots


In [9]:
if PLOT_FIGURES:
    def plot_reliability_curves(method_names, summary_payload, save_path, title):
        plt.figure(figsize=(8, 6))
        plt.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1.2, label="perfect calibration")
        for method_name in method_names:
            bins = summary_payload[method_name]["reliability_bins"]
            xs = [row["mean_confidence"] for row in bins if row["count"] > 0]
            ys = [row["accuracy"] for row in bins if row["count"] > 0]
            plt.plot(xs, ys, marker="o", linewidth=1.8, label=method_name)
        plt.xlabel("Predicted confidence")
        plt.ylabel("Empirical accuracy")
        plt.title(title)
        plt.legend()
        plt.tight_layout()
        plt.savefig(save_path, dpi=200)
        plt.close()


    def plot_risk_coverage(method_names, summary_payload, save_path, title):
        plt.figure(figsize=(8, 6))
        for method_name in method_names:
            curve = summary_payload[method_name]["risk_coverage_curve"]
            plt.plot(curve["coverage"], curve["risk"], linewidth=1.8, label=method_name)
        plt.xlabel("Coverage")
        plt.ylabel("Selective risk")
        plt.title(title)
        plt.legend()
        plt.tight_layout()
        plt.savefig(save_path, dpi=200)
        plt.close()


    plot_methods = ["main_raw_confidence", "main_posthoc_confidence_only", "main_qcur"]
    for tag, run in SEED_RUNS.items():
        phase5_paths = run["phase5_paths"]
        plot_reliability_curves(
            plot_methods,
            run["method_summaries"],
            phase5_paths["qcur_reliability_plot_path"],
            f"Reliability Diagram on PRUGTM-Hybrid Test Set ({tag})",
        )
        plot_risk_coverage(
            plot_methods,
            run["method_summaries"],
            phase5_paths["qcur_risk_coverage_plot_path"],
            f"Risk-Coverage Curves on PRUGTM-Hybrid Test Set ({tag})",
        )
        print(f"{tag}: saved reliability diagram: {phase5_paths['qcur_reliability_plot_path']}")
        print(f"{tag}: saved risk-coverage plot: {phase5_paths['qcur_risk_coverage_plot_path']}")
else:
    print("Skipping plots because PLOT_FIGURES is False.")


seed_42: saved reliability diagram: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_42/figures/qcur_reliability_diagram.png
seed_42: saved risk-coverage plot: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_42/figures/qcur_risk_coverage.png
seed_1337: saved reliability diagram: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_1337/figures/qcur_reliability_diagram.png
seed_1337: saved risk-coverage plot: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliabil

# Save Final Phase 05A PRUGTM-Hybrid Three-Seed Q-CUR Report


In [10]:
SEED_REPORTS = {}

for tag, run in SEED_RUNS.items():
    phase4_paths = run["phase4_paths"]
    phase5_paths = run["phase5_paths"]

    qcur_config = {
        "created_at": now_string(),
        "dataset_release_name": DATASET_RELEASE_NAME,
        "phase5_method_name": PHASE5_METHOD_NAME,
        "phase5_model_variant": PHASE5_MODEL_VARIANT,
        "run_seed": int(run["run_seed"]),
        "seed_tag": tag,
        "phase4_reference_parent_dir": str(PHASE4_REFERENCE_PARENT_DIR),
        "phase4_reference_dir": str(phase4_paths["phase4_dir"]),
        "phase5_output_dir": str(phase5_paths["phase5_seed_dir"]),
        "val_predictions_input_path": str(phase4_paths["val_predictions_input_path"]),
        "test_predictions_input_path": str(phase4_paths["test_predictions_input_path"]),
        "baseline_numeric_features": BASELINE_NUMERIC_FEATURES,
        "qcur_numeric_features": QCUR_NUMERIC_FEATURES,
        "qcur_categorical_features": QCUR_CATEGORICAL_FEATURES,
        "calibration_bins": CALIBRATION_BINS,
        "selective_coverages": SELECTIVE_COVERAGES,
        "enable_optional_hybrid_raw_comparison": ENABLE_OPTIONAL_HYBRID_RAW_COMPARISON,
    }
    write_json(phase5_paths["qcur_config_path"], qcur_config)

    qcur_report = {
        "finished_at": now_string(),
        "phase": "Phase 05A - Four-Seed Q-CUR Reliability Calibration on PRUGTM-Hybrid Main Reference",
        "dataset_release_name": DATASET_RELEASE_NAME,
        "phase5_method_name": PHASE5_METHOD_NAME,
        "phase5_model_variant": PHASE5_MODEL_VARIANT,
        "run_seed": int(run["run_seed"]),
        "seed_tag": tag,
        "source_answer_backbone": run["phase4_final_eval"].get("model_variant"),
        "source_answer_metrics": {
            "validation": run["phase4_val_metrics"],
            "test": run["phase4_test_metrics"],
        },
        "qcur_scope": {
            "frozen_answer_model": True,
            "trained_on": "same-seed validation predictions only",
            "evaluated_on": "same-seed test predictions only",
            "target": "probability that the predicted answer is correct",
        },
        "comparison_methods": run["method_summaries"],
        "slice_metrics_path": str(phase5_paths["qcur_slice_metrics_path"]),
        "method_table_path": str(phase5_paths["qcur_method_table_path"]),
        "val_qcur_predictions_path": str(phase5_paths["qcur_val_predictions_path"]),
        "test_qcur_predictions_path": str(phase5_paths["qcur_test_predictions_path"]),
        "models_path": str(phase5_paths["qcur_models_path"]),
        "reliability_figure_path": str(phase5_paths["qcur_reliability_plot_path"]),
        "risk_coverage_figure_path": str(phase5_paths["qcur_risk_coverage_plot_path"]),
    }
    write_json(run["phase5_paths"]["qcur_report_path"], qcur_report)
    write_json(
        run["phase5_paths"]["qcur_done_path"],
        {
            "finished_at": now_string(),
            "phase": "Phase 05A - Four-Seed Q-CUR Reliability Calibration on PRUGTM-Hybrid Main Reference",
            "phase5_model_variant": PHASE5_MODEL_VARIANT,
            "run_seed": int(run["run_seed"]),
            "seed_tag": tag,
            "qcur_report_path": str(run["phase5_paths"]["qcur_report_path"]),
        },
    )
    SEED_REPORTS[tag] = qcur_report
    print(f"{tag}: Q-CUR config saved: {phase5_paths['qcur_config_path']}")
    print(f"{tag}: Q-CUR report saved: {phase5_paths['qcur_report_path']}")
    print(f"{tag}: Q-CUR done marker saved: {phase5_paths['qcur_done_path']}")

aggregate_config = {
    "created_at": now_string(),
    "dataset_release_name": DATASET_RELEASE_NAME,
    "phase5_method_name": PHASE5_METHOD_NAME,
    "phase5_model_variant": PHASE5_MODEL_VARIANT,
    "run_seeds": [int(seed) for seed in RUN_SEEDS],
    "phase4_reference_parent_dir": str(PHASE4_REFERENCE_PARENT_DIR),
    "phase5_output_dir": str(PHASE5_DIR),
    "baseline_numeric_features": BASELINE_NUMERIC_FEATURES,
    "qcur_numeric_features": QCUR_NUMERIC_FEATURES,
    "qcur_categorical_features": QCUR_CATEGORICAL_FEATURES,
    "calibration_bins": CALIBRATION_BINS,
    "selective_coverages": SELECTIVE_COVERAGES,
    "enable_optional_hybrid_raw_comparison": ENABLE_OPTIONAL_HYBRID_RAW_COMPARISON,
}
write_json(QCUR_CONFIG_PATH, aggregate_config)

qcur_four_seed_report = {
    "finished_at": now_string(),
    "phase": "Phase 05A - Four-Seed Q-CUR Reliability Calibration on PRUGTM-Hybrid Main Reference",
    "dataset_release_name": DATASET_RELEASE_NAME,
    "phase5_method_name": PHASE5_METHOD_NAME,
    "phase5_model_variant": PHASE5_MODEL_VARIANT,
    "run_seeds": [int(seed) for seed in RUN_SEEDS],
    "num_runs": int(len(RUN_SEEDS)),
    "per_seed_reports": {
        tag: str(run["phase5_paths"]["qcur_report_path"])
        for tag, run in SEED_RUNS.items()
    },
    "aggregate_method_summary_path": str(QCUR_AGGREGATE_SUMMARY_PATH),
    "aggregate_method_table_path": str(QCUR_AGGREGATE_METHOD_TABLE_PATH),
    "per_seed_method_table_path": str(QCUR_PER_SEED_METHOD_TABLE_PATH),
    "aggregate_method_rows": AGGREGATE_METHOD_TABLE_DF.to_dict(orient="records"),
    "qcur_scope": {
        "frozen_answer_model": True,
        "trained_on": "per-seed validation predictions only",
        "evaluated_on": "per-seed test predictions only",
        "target": "probability that the predicted answer is correct",
        "aggregation": "mean/std across independent Phase 4 answer-model seeds",
    },
    "next_step": (
        "Use the aggregate mean/std Q-CUR outputs for paper-ready main-reference reliability tables, "
        "then update Phase 05B for four-seed cross-backbone comparison."
    ),
}
write_json(QCUR_REPORT_PATH, qcur_four_seed_report)
write_json(
    QCUR_AGGREGATE_DONE_PATH,
    {
        "finished_at": now_string(),
        "phase": "Phase 05A - Four-Seed Q-CUR Reliability Calibration on PRUGTM-Hybrid Main Reference",
        "phase5_model_variant": PHASE5_MODEL_VARIANT,
        "run_seeds": [int(seed) for seed in RUN_SEEDS],
        "qcur_report_path": str(QCUR_REPORT_PATH),
    },
)

print(json.dumps(qcur_four_seed_report, indent=2)[:6000])
print("Aggregate Q-CUR config saved:", QCUR_CONFIG_PATH)
print("Aggregate Q-CUR report saved:", QCUR_REPORT_PATH)
print("Aggregate Q-CUR done marker saved:", QCUR_AGGREGATE_DONE_PATH)

seed_42: Q-CUR config saved: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_42/phase05a_qcur_config.json
seed_42: Q-CUR report saved: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_42/phase05a_prugtm_hybrid_qcur_report.json
seed_42: Q-CUR done marker saved: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_qcur_main_reference_three_seed/seed_42/done/phase05a_qcur_complete.json
seed_1337: Q-CUR config saved: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5a_best_model_qcur_reliability/btumqa_225k_prugtm_hybrid_q

# Quick Inspect Phase 05A PRUGTM-Hybrid Q-CUR Artifacts


In [11]:
print("Phase 05A PRUGTM-Hybrid Q-CUR output files:\n")
for path in sorted(PHASE5_DIR.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{str(path.relative_to(PHASE5_DIR)):75s} {size_mb:9.2f} MB")

report_preview = read_json(QCUR_REPORT_PATH, {})
if report_preview:
    print("\nRun seeds:", report_preview.get("run_seeds"))
    print("Num runs:", report_preview.get("num_runs"))
    print("Aggregate method table:", report_preview.get("aggregate_method_table_path"))

    aggregate_table = pd.read_csv(QCUR_AGGREGATE_METHOD_TABLE_PATH)
    print("\nAggregate method preview:")
    display_cols = [
        "method",
        "num_runs",
        "accuracy_mean",
        "accuracy_std",
        "macro_f1_mean",
        "macro_f1_std",
        "ece_mean",
        "ece_std",
        "brier_mean",
        "brier_std",
        "nll_mean",
        "nll_std",
        "aurc_mean",
        "aurc_std",
    ]
    print(aggregate_table[display_cols].round(6).to_string(index=False))


Phase 05A PRUGTM-Hybrid Q-CUR output files:

done/phase05a_prugtm_hybrid_qcur_three_seed_complete.json                        0.00 MB
phase05a_prugtm_hybrid_qcur_three_seed_config.json                               0.00 MB
phase05a_prugtm_hybrid_qcur_three_seed_report.json                               0.01 MB
reports/qcur_prugtm_hybrid_three_seed_method_summary.json                        0.00 MB
reports/qcur_prugtm_hybrid_three_seed_method_table.csv                           0.00 MB
reports/qcur_prugtm_hybrid_three_seed_per_seed_method_table.csv                  0.00 MB
seed_1337/done/phase05a_qcur_complete.json                                       0.00 MB
seed_1337/figures/qcur_reliability_diagram.png                                   0.16 MB
seed_1337/figures/qcur_risk_coverage.png                                         0.09 MB
seed_1337/models/qcur_models.joblib                                              0.01 MB
seed_1337/phase05a_prugtm_hybrid_qcur_report.json                